# MNIST Neural Network: Advanced Full OOP Decomposition

**Level**: Advanced  
**Architecture**: Complete object decomposition with Node, Connection, and Layer hierarchy  
**Design Patterns**: Strategy, Composite, Template Method  
**Purpose**: Demonstrate sophisticated OOP design for understanding system architecture

---

## Problem Statement

Build a neural network that learns to classify handwritten digits (0-9).

**Inputs**: 28×28 grayscale images flattened to 784 float values (normalised 0-1)  
**Outputs**: 10 probability values for digits 0-9 (softmax), compared against one-hot encoded labels  
**Task**: Learn the mapping from 784 inputs to 10 outputs through supervised learning

---

## Architecture Overview

This notebook demonstrates **maximum decomposition** - every conceptual element becomes its own class.

```
Level 1 - Atomic Components:
├── ActivationFunction (abstract) → Sigmoid, ReLU, Softmax
├── Node (individual neuron)
└── Connection (weighted link between nodes)

Level 2 - Composite Components:
├── Layer (abstract)
│   ├── InputLayer (no learning)
│   ├── HiddenLayer (full learning)
│   └── OutputLayer (learning + softmax + prediction)

Level 3 - System Components:
├── NeuralNetwork (structure management)
├── Trainer (learning orchestration)
└── Evaluator (performance metrics)
```

### Design Patterns Used

| Pattern | Where Applied | Benefit |
|---------|--------------|--------|
| Strategy | ActivationFunction hierarchy | Swap activations without changing Node |
| Composite | Layer contains Nodes | Treat individual and collection uniformly |
| Template Method | Layer.forward() | Common algorithm, specialised details |
| Facade | NeuralNetwork | Simple interface to complex subsystem |

---

## Trade-offs in This Design

| Choice Made | Alternative | Rationale |
|-------------|-------------|----------|
| Individual Node/Connection objects | Matrix operations | Clarity over performance |
| Explicit connections list | Implicit via weight matrices | Easier to visualise/debug |
| Separate layer types via inheritance | Type flag with conditionals | Open/Closed principle |
| Stateless activation functions | Stateful with caching | Simpler, thread-safe |
| Connection stores references to nodes | Indices into arrays | More Pythonic, easier to follow |

In [ ]:
# Standard library imports only - no external dependencies
from abc import ABC, abstractmethod
from typing import List, Dict, Tuple, Optional
import math
import random

---

## Part 1: Activation Functions (Strategy Pattern)

**Design Pattern**: Strategy Pattern

**Problem**: Nodes need different mathematical transformations, but we want to swap them easily.

**Solution**: Define a common interface (`ActivationFunction`), then create specific strategies (`Sigmoid`, `ReLU`, etc.).

**Benefit**: Node class doesn't care WHICH activation it uses - just that it has one that conforms to the interface.

In [ ]:
class ActivationFunction(ABC):
    """
    Abstract base class for activation functions.
    
    Design Decision: Abstract class ensures all activation functions
    provide the same interface. This enables polymorphism - Node can
    work with ANY activation function without modification.
    
    Alternative: Could use duck typing (no ABC), but explicit interface
    provides better documentation and IDE support.
    """
    
    @abstractmethod
    def activate(self, x: float) -> float:
        """Apply activation function to input."""
        pass
    
    @abstractmethod
    def derivative(self, x: float) -> float:
        """Calculate derivative for backpropagation."""
        pass
    
    @property
    def name(self) -> str:
        """Return class name for display."""
        return self.__class__.__name__
    
    def __str__(self) -> str:
        return self.name

In [ ]:
class Sigmoid(ActivationFunction):
    """
    Sigmoid activation: σ(x) = 1 / (1 + e^(-x))
    
    Properties:
    - Output range: (0, 1)
    - Smooth, differentiable everywhere
    - Can cause vanishing gradient for large |x|
    
    Use case: Output layer for binary classification, or when
    outputs need to be interpreted as probabilities.
    """
    
    def activate(self, x: float) -> float:
        # Clip to prevent overflow in exp
        x = max(-500, min(500, x))
        return 1.0 / (1.0 + math.exp(-x))
    
    def derivative(self, x: float) -> float:
        # Elegant property: σ'(x) = σ(x) * (1 - σ(x))
        s = self.activate(x)
        return s * (1.0 - s)

In [ ]:
class ReLU(ActivationFunction):
    """
    Rectified Linear Unit: f(x) = max(0, x)
    
    Properties:
    - Output range: [0, ∞)
    - Not differentiable at x=0 (we use 0 by convention)
    - Fast to compute
    - Helps avoid vanishing gradient problem
    
    Use case: Hidden layers (most common modern choice)
    """
    
    def activate(self, x: float) -> float:
        return max(0.0, x)
    
    def derivative(self, x: float) -> float:
        return 1.0 if x > 0 else 0.0

In [ ]:
class LeakyReLU(ActivationFunction):
    """
    Leaky ReLU: f(x) = x if x > 0 else alpha * x
    
    Properties:
    - Small negative slope prevents "dying ReLU" problem
    - Typically alpha = 0.01
    
    Use case: When standard ReLU causes too many dead neurons
    """
    
    def __init__(self, alpha: float = 0.01):
        self.alpha = alpha
    
    def activate(self, x: float) -> float:
        return x if x > 0 else self.alpha * x
    
    def derivative(self, x: float) -> float:
        return 1.0 if x > 0 else self.alpha
    
    @property
    def name(self) -> str:
        return f"LeakyReLU(α={self.alpha})"

In [ ]:
class Identity(ActivationFunction):
    """
    Identity activation: f(x) = x
    
    Use case: Input layer (pass-through), or linear output layer
    """
    
    def activate(self, x: float) -> float:
        return x
    
    def derivative(self, x: float) -> float:
        return 1.0

---

## Part 2: Node (Individual Neuron)

**Design Decision**: Composition over Inheritance

Node **has-a** ActivationFunction (composition), not **is-a** ActivationFunction (inheritance).

**Why?** Node and ActivationFunction are fundamentally different concepts:
- Node: Processing unit with state (bias, output, gradient)
- ActivationFunction: Stateless mathematical transformation

### What Node Does

1. Receives weighted sum from connections
2. Adds its bias term
3. Applies activation function
4. Caches output for forward propagation
5. Stores delta for backpropagation

In [ ]:
class Node:
    """
    Represents a single neuron in the neural network.
    
    Design Decisions:
    1. Stores activation function reference (Strategy Pattern)
    2. Caches output (avoid recalculation during backprop)
    3. Stores pre-activation value (needed for derivative)
    4. Stores delta for gradient computation
    5. Single Responsibility: Transform input, nothing more
    
    Alternative: Could merge Node into Layer (simpler but less flexible)
    """
    
    def __init__(self, 
                 node_id: str, 
                 activation_fn: ActivationFunction,
                 bias: float = 0.0):
        """
        Initialise node with activation function and bias.
        
        Args:
            node_id: Unique identifier for debugging/visualisation
            activation_fn: Which activation to use (Strategy)
            bias: Initial bias (typically 0 or small random)
        """
        self.node_id = node_id
        self.activation_fn = activation_fn
        self.bias = bias
        
        # Cached values for forward/backward pass
        self.pre_activation = 0.0  # z = weighted_sum + bias (before activation)
        self.output = 0.0          # a = activation(z)
        self.delta = 0.0           # δ = gradient for backpropagation
    
    def calculate_output(self, input_sum: float) -> float:
        """
        Calculate node output given weighted input sum.
        
        Process:
        1. Add bias to input sum → pre_activation
        2. Apply activation function → output
        3. Cache both values
        4. Return output
        
        Args:
            input_sum: Weighted sum from incoming connections
            
        Returns:
            Activated output value
        """
        self.pre_activation = input_sum + self.bias
        self.output = self.activation_fn.activate(self.pre_activation)
        return self.output
    
    def get_output(self) -> float:
        """Return cached output from last forward pass."""
        return self.output
    
    def get_activation_derivative(self) -> float:
        """Calculate activation derivative at cached pre-activation value."""
        return self.activation_fn.derivative(self.pre_activation)
    
    def set_delta(self, delta: float):
        """Set delta for backpropagation."""
        self.delta = delta
    
    def get_delta(self) -> float:
        """Return delta for backpropagation."""
        return self.delta
    
    def update_bias(self, learning_rate: float):
        """
        Update bias using gradient descent.
        
        Formula: bias -= learning_rate * delta
        """
        self.bias -= learning_rate * self.delta
    
    def __str__(self) -> str:
        return f"Node({self.node_id}, {self.activation_fn}, bias={self.bias:.4f})"
    
    def __repr__(self) -> str:
        return self.__str__()

---

## Part 3: Connection (Weighted Link)

**Design Decision**: Separate Relationship from Entity

**Question**: Why not store connections inside Node?

**Answer**: Separation of concerns
- Node: Responsible for computation
- Connection: Responsible for network topology and weight management

**Benefits**:
1. Can change network structure without touching Node code
2. Can implement different connection patterns (sparse, skip connections, etc.)
3. Easier to test Node and Connection independently
4. Clearer mental model of the network

In [ ]:
class Connection:
    """
    Represents a weighted connection between two nodes.
    
    Design Decision: Connection is separate from Node
    - Node handles computation (single responsibility)
    - Connection handles relationships and weight learning
    - Easier to modify network topology without changing Node
    
    Alternative: Store outgoing connections in Node
    - Simpler navigation but violates separation of concerns
    - Harder to implement advanced topologies
    """
    
    def __init__(self, 
                 from_node: Node, 
                 to_node: Node,
                 weight: Optional[float] = None):
        """
        Initialise connection between two nodes.
        
        Args:
            from_node: Source node (previous layer)
            to_node: Destination node (next layer)
            weight: Initial weight (if None, use Xavier initialisation)
        """
        self.from_node = from_node
        self.to_node = to_node
        
        # Xavier/Glorot initialisation if not provided
        if weight is None:
            # Scale based on typical fan-in (approximation)
            self.weight = random.gauss(0, 0.5)
        else:
            self.weight = weight
        
        self.gradient = 0.0  # Accumulated gradient for this weight
    
    def forward(self) -> float:
        """
        Calculate weighted output for forward pass.
        
        Returns:
            from_node.output * weight
        """
        return self.from_node.get_output() * self.weight
    
    def calculate_gradient(self):
        """
        Calculate gradient for this connection's weight.
        
        Formula: gradient = from_node.output * to_node.delta
        
        This is the chain rule: dL/dw = dL/da * da/dz * dz/dw
        where dz/dw = from_node.output (the input to this weight)
        """
        self.gradient = self.from_node.get_output() * self.to_node.get_delta()
    
    def update_weight(self, learning_rate: float):
        """
        Update weight using gradient descent.
        
        Formula: weight -= learning_rate * gradient
        """
        self.weight -= learning_rate * self.gradient
    
    def get_weight(self) -> float:
        """Return current weight value."""
        return self.weight
    
    def __str__(self) -> str:
        return f"Connection({self.from_node.node_id} → {self.to_node.node_id}, w={self.weight:.4f})"
    
    def __repr__(self) -> str:
        return self.__str__()

---

## Part 4: Layer Abstraction (Template Method + Composite)

**Design Pattern**: Template Method
- Base class defines algorithm structure (`forward`)
- Subclasses specialise specific behaviours

**Design Pattern**: Composite
- Layer contains multiple Nodes
- Treats the collection uniformly

### Layer Hierarchy

```
Layer (abstract)
  ├── InputLayer (pass-through, no learning)
  ├── HiddenLayer (full learning, typically ReLU)
  └── OutputLayer (softmax + prediction)
```

**Why Three Layer Types?**

Could have used a single Layer class with type flags and conditionals. Problems:
- Violates Single Responsibility (one class knows too much)
- Hard to extend (adding new type requires modifying existing code)
- Violates Open/Closed Principle

Better approach: Inheritance with specialisation

In [ ]:
class Layer(ABC):
    """
    Abstract base class for neural network layers.
    
    Design Pattern: Template Method
    - forward() provides common algorithm structure
    - Subclasses override specific behaviours
    
    Design Pattern: Composite
    - Layer contains collection of Nodes
    - Operations applied to all nodes uniformly
    """
    
    def __init__(self, 
                 layer_id: str, 
                 num_nodes: int,
                 activation_fn: ActivationFunction,
                 layer_type: str):
        """
        Initialise layer with specified number of nodes.
        
        Args:
            layer_id: Identifier for debugging
            num_nodes: How many nodes in this layer
            activation_fn: Activation function for all nodes
            layer_type: 'input', 'hidden', or 'output'
        """
        self.layer_id = layer_id
        self.layer_type = layer_type
        self.nodes: List[Node] = []
        
        # Create nodes with small random biases
        for i in range(num_nodes):
            node_id = f"{layer_id}_n{i}"
            bias = random.gauss(0, 0.01)
            node = Node(node_id, activation_fn, bias)
            self.nodes.append(node)
    
    def forward(self, inputs: List[float]) -> List[float]:
        """
        Process inputs through all nodes.
        
        Template method - same algorithm for all layer types.
        Each node receives its corresponding input value.
        
        Args:
            inputs: List of input values (one per node)
            
        Returns:
            List of output values (one per node)
        """
        if len(inputs) != len(self.nodes):
            raise ValueError(f"Layer {self.layer_id}: expected {len(self.nodes)} inputs, got {len(inputs)}")
        
        for node, input_val in zip(self.nodes, inputs):
            node.calculate_output(input_val)
        
        return self.get_outputs()
    
    def get_outputs(self) -> List[float]:
        """Return outputs from all nodes."""
        return [node.get_output() for node in self.nodes]
    
    def get_num_nodes(self) -> int:
        """Return number of nodes in layer."""
        return len(self.nodes)
    
    def update_biases(self, learning_rate: float):
        """Update all node biases using gradient descent."""
        for node in self.nodes:
            node.update_bias(learning_rate)
    
    @abstractmethod
    def describe(self) -> str:
        """Return description of layer (subclasses implement)."""
        pass
    
    def __str__(self) -> str:
        return f"{self.layer_type.capitalize()}Layer({self.layer_id}, {len(self.nodes)} nodes)"
    
    def __repr__(self) -> str:
        return self.__str__()

In [ ]:
class InputLayer(Layer):
    """
    Input layer - receives raw data.
    
    Special characteristics:
    - No learning (no backpropagation through this layer)
    - Uses Identity activation (pass-through)
    - Just holds input values for connections to read
    
    For MNIST: 784 nodes (28×28 pixels)
    """
    
    def __init__(self, layer_id: str, input_size: int):
        # Identity activation - just passes values through
        super().__init__(layer_id, input_size, Identity(), "input")
        self.input_size = input_size
        
        # Input layer biases should be zero (pure pass-through)
        for node in self.nodes:
            node.bias = 0.0
    
    def describe(self) -> str:
        return f"InputLayer: {self.input_size} features"

In [ ]:
class HiddenLayer(Layer):
    """
    Hidden layer - internal processing.
    
    Special characteristics:
    - Full learning capability
    - Typically uses ReLU activation
    - Can have multiple hidden layers stacked
    
    Purpose: Learn increasingly abstract representations
    """
    
    def __init__(self, 
                 layer_id: str, 
                 num_nodes: int,
                 activation_fn: Optional[ActivationFunction] = None):
        if activation_fn is None:
            activation_fn = ReLU()  # Default to ReLU
        
        super().__init__(layer_id, num_nodes, activation_fn, "hidden")
    
    def describe(self) -> str:
        act_name = self.nodes[0].activation_fn.name
        return f"HiddenLayer: {len(self.nodes)} nodes, {act_name}"

In [ ]:
class OutputLayer(Layer):
    """
    Output layer - produces predictions.
    
    Special characteristics:
    - Applies softmax normalisation across all outputs
    - Number of nodes = number of classes
    - Has prediction method (argmax)
    
    For MNIST: 10 nodes (digits 0-9)
    
    Note: Softmax is applied at the layer level (not per-node)
    because it requires all values to compute normalisation.
    """
    
    def __init__(self, layer_id: str, num_classes: int):
        # Use Identity in nodes - we apply softmax at layer level
        super().__init__(layer_id, num_classes, Identity(), "output")
        self.num_classes = num_classes
    
    def forward(self, inputs: List[float]) -> List[float]:
        """
        Forward pass with softmax normalisation.
        
        Process:
        1. Compute raw outputs (logits) from each node
        2. Apply softmax across all outputs
        3. Store normalised values back in nodes
        """
        # Step 1: Get raw outputs (logits)
        super().forward(inputs)
        logits = [node.output for node in self.nodes]
        
        # Step 2: Apply softmax
        probabilities = self._softmax(logits)
        
        # Step 3: Store back in nodes (for backprop to read)
        for node, prob in zip(self.nodes, probabilities):
            node.output = prob
        
        return probabilities
    
    def _softmax(self, logits: List[float]) -> List[float]:
        """
        Apply softmax to convert logits to probabilities.
        
        softmax(x_i) = exp(x_i) / sum(exp(x_j) for all j)
        
        Uses max-subtraction for numerical stability.
        """
        # Subtract max for numerical stability (prevents overflow)
        max_logit = max(logits)
        exp_values = [math.exp(x - max_logit) for x in logits]
        total = sum(exp_values)
        return [x / total for x in exp_values]
    
    def get_prediction(self) -> int:
        """
        Get predicted class (index of highest probability).
        """
        outputs = self.get_outputs()
        return outputs.index(max(outputs))
    
    def describe(self) -> str:
        return f"OutputLayer: {self.num_classes} classes, Softmax"

---

## Part 5: Neural Network (Facade Pattern)

**Design Pattern**: Facade
- Provides simple interface to complex subsystem
- Hides Layer/Node/Connection complexity from users

**Responsibilities**:
- Build network from architecture specification
- Manage layers and connections
- Coordinate forward and backward passes

**Not Responsible For**:
- Data loading (DataProcessor's job)
- Training loop (Trainer's job)
- Evaluation metrics (Evaluator's job)

In [ ]:
class NeuralNetwork:
    """
    Complete neural network structure.
    
    Design Pattern: Facade
    - Simple interface: forward(), backward(), update()
    - Hides complex Layer/Node/Connection interactions
    
    Design: Composition of layers and connections
    - Manages ordered list of layers
    - Creates fully-connected connections between adjacent layers
    - Coordinates forward and backward passes
    """
    
    def __init__(self, network_id: str, architecture: Dict):
        """
        Build network from architecture specification.
        
        Args:
            network_id: Identifier for this network
            architecture: Dict with keys:
                - 'input_size': int
                - 'hidden_layers': List[int] (sizes)
                - 'output_size': int
        
        Example:
            architecture = {
                'input_size': 784,
                'hidden_layers': [128, 64],
                'output_size': 10
            }
        """
        self.network_id = network_id
        self.architecture = architecture
        self.layers: List[Layer] = []
        self.connections: List[List[Connection]] = []  # connections[i] = connections from layer i to layer i+1
        
        self._build_network()
    
    def _build_network(self):
        """
        Construct layers and connections from architecture spec.
        """
        # Step 1: Create input layer
        input_layer = InputLayer("input", self.architecture['input_size'])
        self.layers.append(input_layer)
        
        # Step 2: Create hidden layers
        for i, size in enumerate(self.architecture['hidden_layers']):
            hidden_layer = HiddenLayer(f"hidden{i+1}", size)
            self.layers.append(hidden_layer)
        
        # Step 3: Create output layer
        output_layer = OutputLayer("output", self.architecture['output_size'])
        self.layers.append(output_layer)
        
        # Step 4: Connect adjacent layers (fully connected)
        for i in range(len(self.layers) - 1):
            self._connect_layers(self.layers[i], self.layers[i + 1])
    
    def _connect_layers(self, from_layer: Layer, to_layer: Layer):
        """
        Create fully-connected links between two layers.
        
        Every node in from_layer connects to every node in to_layer.
        """
        layer_connections = []
        
        for from_node in from_layer.nodes:
            for to_node in to_layer.nodes:
                connection = Connection(from_node, to_node)
                layer_connections.append(connection)
        
        self.connections.append(layer_connections)
    
    def forward(self, inputs: List[float]) -> List[float]:
        """
        Forward pass through the network.
        
        Process:
        1. Feed inputs to input layer
        2. For each subsequent layer:
           a. Gather weighted sums via connections
           b. Pass to layer for processing
        3. Return output layer's probabilities
        """
        # Input layer just stores the values
        self.layers[0].forward(inputs)
        
        # Process through hidden and output layers
        for layer_idx in range(1, len(self.layers)):
            next_layer = self.layers[layer_idx]
            
            # Calculate weighted sums for each node in next layer
            layer_inputs = [0.0] * next_layer.get_num_nodes()
            
            # Sum contributions from all incoming connections
            for connection in self.connections[layer_idx - 1]:
                to_idx = next_layer.nodes.index(connection.to_node)
                layer_inputs[to_idx] += connection.forward()
            
            # Process through the layer
            next_layer.forward(layer_inputs)
        
        return self.layers[-1].get_outputs()
    
    def backward(self, targets: List[float]):
        """
        Backward pass (backpropagation) to compute gradients.
        
        Args:
            targets: One-hot encoded target labels
        
        Process:
        1. Compute output layer deltas (prediction - target)
        2. Propagate deltas backward through layers
        3. Compute connection gradients
        """
        # Step 1: Output layer deltas
        # For softmax + cross-entropy, delta = output - target
        output_layer = self.layers[-1]
        for i, node in enumerate(output_layer.nodes):
            delta = node.get_output() - targets[i]
            node.set_delta(delta)
        
        # Step 2: Hidden layer deltas (back to front)
        for layer_idx in range(len(self.layers) - 2, 0, -1):
            current_layer = self.layers[layer_idx]
            connections_to_next = self.connections[layer_idx]
            
            for node in current_layer.nodes:
                # Sum of (weight * downstream_delta) for all outgoing connections
                weighted_delta_sum = 0.0
                for conn in connections_to_next:
                    if conn.from_node == node:
                        weighted_delta_sum += conn.weight * conn.to_node.get_delta()
                
                # Multiply by activation derivative
                delta = weighted_delta_sum * node.get_activation_derivative()
                node.set_delta(delta)
        
        # Step 3: Compute connection gradients
        for layer_connections in self.connections:
            for connection in layer_connections:
                connection.calculate_gradient()
    
    def update(self, learning_rate: float):
        """
        Update all weights and biases using computed gradients.
        """
        # Update connection weights
        for layer_connections in self.connections:
            for connection in layer_connections:
                connection.update_weight(learning_rate)
        
        # Update node biases (skip input layer)
        for layer in self.layers[1:]:
            layer.update_biases(learning_rate)
    
    def predict(self, inputs: List[float]) -> int:
        """
        Make prediction for given input.
        """
        self.forward(inputs)
        return self.layers[-1].get_prediction()
    
    def predict_proba(self, inputs: List[float]) -> List[float]:
        """
        Get probability distribution for given input.
        """
        return self.forward(inputs)
    
    def print_architecture(self):
        """
        Display network architecture.
        """
        print(f"\nNetwork: {self.network_id}")
        print("=" * 50)
        
        for i, layer in enumerate(self.layers):
            print(f"Layer {i}: {layer.describe()}")
            
            if i < len(self.connections):
                num_connections = len(self.connections[i])
                print(f"  → {num_connections} connections to next layer")
        
        print("=" * 50)
        
        # Count parameters
        total_weights = sum(len(layer_conns) for layer_conns in self.connections)
        total_biases = sum(layer.get_num_nodes() for layer in self.layers[1:])  # Skip input
        total_params = total_weights + total_biases
        
        print(f"Total parameters: {total_params:,}")
        print(f"  Weights: {total_weights:,}")
        print(f"  Biases: {total_biases:,}")
    
    def summary(self) -> str:
        """
        Return brief architecture summary.
        """
        layer_sizes = [layer.get_num_nodes() for layer in self.layers]
        return f"{self.network_id}: {' → '.join(map(str, layer_sizes))}"

---

## Part 6: Trainer (Separation of Concerns)

**Design Decision**: Separate training logic from network structure.

**Why?**
- NeuralNetwork knows HOW to compute (forward, backward, update)
- Trainer knows WHEN to compute (epochs, batches, learning rate schedules)

**Benefits**:
- Can swap training strategies without changing network
- Easier to test network structure independently
- Can add features like learning rate decay, early stopping, etc.

In [ ]:
class Trainer:
    """
    Manages the training process for a neural network.
    
    Separation of Concerns:
    - NeuralNetwork: Structure and computation
    - Trainer: Learning orchestration
    
    Responsibilities:
    - Run training loop (epochs)
    - Compute and track loss
    - Manage learning rate
    - Report progress
    """
    
    def __init__(self, 
                 network: NeuralNetwork,
                 learning_rate: float = 0.01):
        """
        Initialise trainer with network and learning rate.
        """
        self.network = network
        self.learning_rate = learning_rate
        self.history = {'loss': [], 'accuracy': []}
    
    def _cross_entropy_loss(self, predictions: List[float], targets: List[float]) -> float:
        """
        Compute cross-entropy loss.
        
        L = -sum(target_i * log(pred_i))
        """
        loss = 0.0
        for pred, target in zip(predictions, targets):
            if target > 0:  # Only non-zero targets contribute
                # Clip prediction to avoid log(0)
                pred = max(pred, 1e-15)
                loss -= target * math.log(pred)
        return loss
    
    def _one_hot(self, label: int, num_classes: int) -> List[float]:
        """
        Convert integer label to one-hot encoding.
        """
        encoding = [0.0] * num_classes
        encoding[label] = 1.0
        return encoding
    
    def train_epoch(self, 
                    X: List[List[float]], 
                    y: List[int]) -> Tuple[float, float]:
        """
        Train for one epoch.
        
        Args:
            X: List of input samples
            y: List of integer labels
            
        Returns:
            Tuple of (average_loss, accuracy)
        """
        num_classes = self.network.architecture['output_size']
        total_loss = 0.0
        correct = 0
        
        for inputs, label in zip(X, y):
            # Forward pass
            predictions = self.network.forward(inputs)
            targets = self._one_hot(label, num_classes)
            
            # Compute loss
            total_loss += self._cross_entropy_loss(predictions, targets)
            
            # Check accuracy
            predicted_class = predictions.index(max(predictions))
            if predicted_class == label:
                correct += 1
            
            # Backward pass
            self.network.backward(targets)
            
            # Update weights
            self.network.update(self.learning_rate)
        
        avg_loss = total_loss / len(X)
        accuracy = correct / len(X)
        
        return avg_loss, accuracy
    
    def fit(self, 
            X: List[List[float]], 
            y: List[int],
            epochs: int = 10,
            verbose: bool = True) -> Dict:
        """
        Train the network for specified number of epochs.
        
        Args:
            X: Training inputs
            y: Training labels
            epochs: Number of training epochs
            verbose: Whether to print progress
            
        Returns:
            Training history dictionary
        """
        for epoch in range(epochs):
            loss, accuracy = self.train_epoch(X, y)
            
            self.history['loss'].append(loss)
            self.history['accuracy'].append(accuracy)
            
            if verbose:
                print(f"Epoch {epoch + 1}/{epochs} - loss: {loss:.4f} - accuracy: {accuracy:.4f}")
        
        return self.history

---

## Part 7: Evaluator (Single Responsibility)

**Design Decision**: Separate evaluation from training.

**Why?**
- Single Responsibility: Evaluator only computes metrics
- Can add new metrics without touching Trainer
- Can evaluate on different datasets easily

In [ ]:
class Evaluator:
    """
    Evaluates network performance.
    
    Separation of Concerns:
    - Trainer: Learning process
    - Evaluator: Performance measurement
    
    Responsibilities:
    - Compute accuracy
    - Compute loss
    - Generate confusion information
    """
    
    def __init__(self, network: NeuralNetwork):
        self.network = network
    
    def evaluate(self, 
                 X: List[List[float]], 
                 y: List[int]) -> Dict:
        """
        Evaluate network on a dataset.
        
        Returns:
            Dictionary with 'accuracy', 'loss', and 'predictions'
        """
        num_classes = self.network.architecture['output_size']
        predictions = []
        correct = 0
        total_loss = 0.0
        
        for inputs, label in zip(X, y):
            probs = self.network.predict_proba(inputs)
            pred = probs.index(max(probs))
            predictions.append(pred)
            
            if pred == label:
                correct += 1
            
            # Compute loss
            target_prob = max(probs[label], 1e-15)
            total_loss -= math.log(target_prob)
        
        return {
            'accuracy': correct / len(X),
            'loss': total_loss / len(X),
            'predictions': predictions
        }
    
    def accuracy(self, X: List[List[float]], y: List[int]) -> float:
        """
        Compute accuracy only.
        """
        correct = sum(1 for inputs, label in zip(X, y) 
                     if self.network.predict(inputs) == label)
        return correct / len(X)

---

## Demonstration

Let's test this architecture with synthetic data.

In [ ]:
def generate_synthetic_data(num_samples: int, 
                            input_size: int, 
                            num_classes: int,
                            seed: int = 42) -> Tuple[List[List[float]], List[int]]:
    """
    Generate synthetic classification data.
    
    Labels determined by which chunk of the input vector has the highest sum.
    This creates a learnable but non-trivial pattern.
    """
    random.seed(seed)
    X = []
    y = []
    
    chunk_size = input_size // num_classes
    
    for _ in range(num_samples):
        # Random input
        sample = [random.random() for _ in range(input_size)]
        
        # Compute chunk sums
        chunk_sums = []
        for i in range(num_classes):
            start = i * chunk_size
            end = start + chunk_size
            chunk_sums.append(sum(sample[start:end]))
        
        # Label is the chunk with highest sum
        label = chunk_sums.index(max(chunk_sums))
        
        X.append(sample)
        y.append(label)
    
    return X, y

# Generate data
INPUT_SIZE = 20
NUM_CLASSES = 4
HIDDEN_SIZE = 12

X_train, y_train = generate_synthetic_data(500, INPUT_SIZE, NUM_CLASSES, seed=42)
X_test, y_test = generate_synthetic_data(100, INPUT_SIZE, NUM_CLASSES, seed=123)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Input size: {INPUT_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")

In [ ]:
# Build and inspect network architecture
architecture = {
    'input_size': INPUT_SIZE,
    'hidden_layers': [HIDDEN_SIZE],
    'output_size': NUM_CLASSES
}

network = NeuralNetwork("advanced_demo", architecture)
network.print_architecture()

In [ ]:
# Inspect individual components
print("\n--- Component Inspection ---\n")

# Layers
print("Layers:")
for layer in network.layers:
    print(f"  {layer}")

# Sample nodes
print(f"\nSample node from hidden layer:")
print(f"  {network.layers[1].nodes[0]}")

# Sample connections
print(f"\nSample connections (first 3 from input to hidden):")
for conn in network.connections[0][:3]:
    print(f"  {conn}")

In [ ]:
# Test forward pass before training
sample_input = X_train[0]
sample_label = y_train[0]

output = network.forward(sample_input)
prediction = network.predict(sample_input)

print("Before Training:")
print(f"  Sample label: {sample_label}")
print(f"  Prediction: {prediction}")
print(f"  Output probabilities: {[f'{p:.4f}' for p in output]}")

In [ ]:
# Train the network
trainer = Trainer(network, learning_rate=0.1)

print("Training...\n")
history = trainer.fit(X_train, y_train, epochs=20, verbose=True)

In [ ]:
# Evaluate on test set
evaluator = Evaluator(network)
results = evaluator.evaluate(X_test, y_test)

print("\n--- Test Set Evaluation ---")
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Loss: {results['loss']:.4f}")

In [ ]:
# Test forward pass after training
output = network.forward(sample_input)
prediction = network.predict(sample_input)

print("\nAfter Training:")
print(f"  Sample label: {sample_label}")
print(f"  Prediction: {prediction}")
print(f"  Output probabilities: {[f'{p:.4f}' for p in output]}")
print(f"  Correct: {prediction == sample_label}")

---

## Architecture Comparison

How does this Advanced approach compare to simpler designs?

| Aspect | Beginner (Procedural) | Intermediate (Layer-Based) | Advanced (Full OOP) |
|--------|----------------------|---------------------------|--------------------|
| **Classes** | 0-1 | 4-6 | 10+ |
| **Granularity** | Functions | Layers | Nodes + Connections |
| **Testability** | Low (integration only) | Medium | High (unit tests easy) |
| **Flexibility** | Low | Medium | High |
| **Performance** | Best | Good | Slowest |
| **Learning Curve** | Low | Medium | High |
| **Best For** | Learning concepts | Practical use | Understanding architecture |

### When to Use Each Approach

**Beginner (Procedural)**:
- Learning how neural networks compute
- Quick prototyping
- Understanding the mathematics

**Intermediate (Layer-Based)**:
- Building practical networks
- Balance of clarity and usability
- When you need to modify layer behaviour

**Advanced (Full OOP)**:
- Learning software architecture patterns
- When you need maximum flexibility
- Research on novel connection patterns
- Educational demonstrations

---

## Design Patterns Summary

### 1. Strategy Pattern (ActivationFunction)

```
ActivationFunction (interface)
    ├── Sigmoid
    ├── ReLU  
    ├── LeakyReLU
    └── Identity
```

**Benefit**: Swap activations without changing Node class.

### 2. Composite Pattern (Layer contains Nodes)

```
Layer
    └── nodes: List[Node]
```

**Benefit**: Operations apply uniformly to all nodes.

### 3. Template Method (Layer.forward)

```
Layer.forward()  ← Template method (same for all)
    ├── InputLayer: pass-through
    ├── HiddenLayer: ReLU activation
    └── OutputLayer: adds softmax
```

**Benefit**: Common algorithm, specialised details in subclasses.

### 4. Facade Pattern (NeuralNetwork)

```
NeuralNetwork
    ├── forward()   ← Simple interface
    ├── backward()
    └── update()
        └── Hides: Layer, Node, Connection complexity
```

**Benefit**: Users interact with simple interface, complexity hidden.

---

## Trade-offs Revisited

**Chose clarity over performance**:
- Python loops instead of vectorised operations
- Individual Node/Connection objects
- Explicit structure over implicit

**Why?** This notebook is for learning Software Architecture. Production code would optimise using NumPy arrays, but that obscures the object relationships we're studying.

---

## Extension Ideas

This architecture makes it easy to add:

1. **Different connection patterns**: Modify `_connect_layers()` for sparse connections
2. **Skip connections**: Add connections between non-adjacent layers
3. **Dropout**: Add `Dropout` layer type
4. **Different optimisers**: Create `Optimiser` hierarchy (SGD, Adam, etc.)
5. **Regularisation**: Add weight decay to `Connection.update_weight()`

---

## Reflection Questions

1. Why is `ActivationFunction` an abstract class instead of using duck typing?
2. Why separate `Connection` from `Node` rather than storing weights in nodes?
3. What would happen if we used one `Layer` class with type checking instead of inheritance?
4. How would you modify this architecture to support recurrent connections?
5. What are the trade-offs between this design and using NumPy arrays directly?

---

**End of Advanced Full OOP Notebook**